# **MODEL EXPERIMENTATION**
with GCP Integration


In [1]:
#imports 
import sys

sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scripts.plot_utils import (
    set_theme,
    plot_grid,
    plot_barplot,
    plot_barplot_grid,
    plot_histogram,
    plot_numeric_x_numeric_grid,
    plot_numeric_x_across_categories_grid,
    plot_all_numeric_by_base_category_grid,
    plot_categorical_x_categorical_grid,
)
from scripts.gcs_utils import (
    log_dataset_to_gcs,
    log_pipeline_run,
    register_vertex_dataset,
)
from kfp import compiler
from google.cloud import aiplatform

# Components
from vertex.components import (
    load_validate_data,
    split_data,
    oversample_training,
    fit_apply_preprocessing_v1,
    apply_preprocessing_v1,
    train_model,
    evaluate_model,
)

# Pipelines
from vertex.pipelines import (
    preprocessing_pipeline,
    training_pipeline,
    training_pipeline_no_oversample,
)


E0000 00:00:1779648769.332783 9664159 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1779648769.332804 9664159 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1779648769.332805 9664159 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1779648769.332813 9664159 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1779648769.332814 9664159 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [2]:
# variable declarations
TARGET_COL = "readmission_within_30_days"
ID_COL = "patient_id"
PROJECT_ID = "readmission-543-project"
LOCATION = "us-central1"
BUCKET_ROOT_URI = "gs://readmissions_bucket_v2"


---
## KFP Preprocessing Pipeline

Self-contained KFP v2 components for the preprocessing pipeline.  
Order: `load_validate_data` → `split_data` → `oversample_training` → `fit_apply_preprocessing` → `apply_preprocessing`

In [3]:
# Components are defined in vertex/components/ and imported above:
#   load_validate_data                          ← ingest.py
#   split_data                                  ← split.py
#   oversample_training                         ← oversample.py
#   fit_apply_preprocessing_v1                  ← preprocessing.py
#   apply_preprocessing_v1                      ← preprocessing.py
#   train_model                                 ← train.py
#   evaluate_model                              ← evaluate.py


In [4]:
PREPROCESSING_PIPELINE_JSON = "../vertex/pipelines/readmissions_preprocessing_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=preprocessing_pipeline,
    package_path=PREPROCESSING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {PREPROCESSING_PIPELINE_JSON}")


Pipeline compiled to ../vertex/pipelines/readmissions_preprocessing_pipeline.json


---
## KFP Training Pipeline

`train_model` and `evaluate_model` components extending the preprocessing pipeline.  
Order: `...preprocessing...` → `train_model` → `evaluate_model`

- **`model_type`**: `"logistic"` | `"random_forest"` | `"xgboost"`  
- **`hyperparams_json`**: JSON string of kwargs passed to the chosen estimator (e.g. `'{"n_estimators": 200, "max_depth": 5}'`)

In [5]:
TRAINING_PIPELINE_JSON = "../vertex/pipelines/readmissions_training_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=training_pipeline,
    package_path=TRAINING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {TRAINING_PIPELINE_JSON}")


Pipeline compiled to ../vertex/pipelines/readmissions_training_pipeline.json


---
## Pipeline Submission with Vertex AI Datasets

Register the training CSV as a managed Vertex AI Dataset, then submit the training pipeline.  
The dataset resource name flows through `load_validate_data` metadata → Vertex ML Metadata, giving a native console lineage graph: **Dataset → Pipeline Run → Model**.

- To reuse an existing dataset instead of creating a new one, replace `TabularDataset.create(...)` with `aiplatform.TabularDataset(dataset_name="projects/.../datasets/...")`.

In [6]:
from pathlib import Path

RAW_TRAIN_PATH = Path("../data/raw/healthcare_readmissions_dataset_train.csv")
DATASET_VERSION = "v0.0"
EXPERIMENT_NAME = "readmissions-model-exp"

log_dataset_to_gcs(
    DATASET_LOCAL_PATH=RAW_TRAIN_PATH,
    VERSION_ID=DATASET_VERSION,
    BUCKET_ROOT_URI=BUCKET_ROOT_URI,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    log_experiment=True,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    description="Raw training data",
    tags=["raw"],
    resume_run=True,
)


Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v0.0/train.csv
Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v0.0/manifest.json


Logged dataset version to Vertex Experiments: readmissions-model-exp / readmissions-data-v0-0


In [7]:
DATASET_VERSION = "v0.0"
dataset, DATASET_GCS_URI = register_vertex_dataset(
    dataset_version=DATASET_VERSION,
    bucket_root_uri=BUCKET_ROOT_URI,
    project_id=PROJECT_ID,
    location=LOCATION,
)
DATASET_RESOURCE_NAME = dataset.resource_name


Reusing existing dataset: projects/182027088454/locations/us-central1/datasets/1513965389040582656
Dataset registered : projects/182027088454/locations/us-central1/datasets/1513965389040582656
GCS source         : gs://readmissions_bucket_v2/datasets/readmissions/v0.0/train.csv


In [8]:
# Submit training pipeline, then log the run (with eval metrics) to Vertex Experiments.
MODEL_VERSION = "v0"
EXPERIMENT_NAME = "readmissions-model-exp"

job = aiplatform.PipelineJob(
    display_name=f"readmissions-training-{DATASET_VERSION}-{MODEL_VERSION}",
    template_path=TRAINING_PIPELINE_JSON,
    pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
    parameter_values={
        "dataset_gcs_uri": DATASET_GCS_URI,
        "dataset_version": DATASET_VERSION,
        "model_type": "xgboost",
        "hyperparams_json": "{}",
    },
)

job.submit()
print(f"Pipeline submitted: {job.display_name}")

log_pipeline_run(
    pipeline_job=job,
    dataset_version=DATASET_VERSION,
    training_dataset_path=DATASET_GCS_URI,
    model_version=MODEL_VERSION,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    wait_for_completion=True,
)


Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524115259
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524115259')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/readmissions-training-pipeline-20260524115259?project=182027088454
Pipeline submitted: readmissions-training-v0.0-v0


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/readmissions-model-exp-pipeline-run-v0-20260524185300 to Experiment: readmissions-model-exp


Waiting for pipeline to complete...
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524115259
Logged eval metrics: {'val_precision': 0.6915254237288135, 'val_pr_auc': 0.7864636123279664, 'val_recall': 0.7311827956989247, 'val_roc_auc': 0.9442192501975507, 'val_f1': 0.710801393728223}
Logged pipeline run to Vertex Experiments: readmissions-model-exp / pipeline-run-v0-20260524185300


---
## Dataset v1.0 — Outlier Removal

Remove age and BMI outliers identified during EDA (z-score thresholds: age > 3, BMI > 3.5), save the cleaned CSV locally, and register it as a new versioned dataset in GCS and Vertex AI.


In [9]:
raw_df = pd.read_csv(
    "../data/raw/healthcare_readmissions_dataset_train.csv",
    keep_default_na=False,
    na_values=[""],
)

raw_df.head()

,PatientID,Age,Gender,Ethnicity,Hospital ID,Height (m),Smoker,BMI,Weight (kg),Adjusted Weight (kg),Has Diabetes,Has Hypertension,Exercise Frequency,Diet Type,Number of Prior Visits,Medications Prescribed,Length of Stay,Type of Treatment,Readmission within 30 Days
0,1000000,23,Female,African American,Hosp2,1.6,False,25.0,64.0,63.283346,0,0,Regular,High-fat,3.0,3.0,0,None,0
1,1000002,56,Female,Hispanic,Hosp3,1.8,True,27.0,87.5,87.678859,0,0,Regular,High-fat,2.0,NaN,2,None,0
2,1000003,28,Male,African American,Hosp1,1.8,False,35.0,113.4,113.497844,0,1,None,Other,NaN,2.0,5,None,0
3,1000004,70,Female,Caucasian,Hosp2,1.8,False,27.7,89.7,89.717694,0,0,None,Other,3.0,NaN,0,Major Surgery,0
4,1000005,48,Female,Hispanic,Hosp1,1.9,False,22.4,80.9,80.528927,0,0,Occasional,High-fat,7.0,5.0,7,Major Surgery,1


In [10]:
import scipy.stats as stats
import numpy as np
# age outlier checking
threshold = 3
age_z_scores = np.abs(stats.zscore(raw_df["Age"]))
age_outliers = np.where(age_z_scores > threshold)[0]
print(f"Identified {len(age_outliers)} age outliers at threshold {threshold}:")

display(raw_df.loc[age_outliers, "Age"])


Identified 80 age outliers at threshold 3:


122     158
136     167
253     126
535     147
547     120
       ... 
7251    136
7589    165
7814    195
7976    145
8027    156
Name: Age, Length: 80, dtype: int64

In [11]:
df_transformed = raw_df.drop(index=age_outliers).reset_index(drop=True)

In [12]:
# looking into potential bmi outliers

threshold = 3.5
z_scores = np.abs(stats.zscore(df_transformed['BMI']))
bmi_outliers = np.where(z_scores > threshold)[0]
print(f"Identified {len(bmi_outliers)} BMI outliers at threshold {threshold}:")

display(df_transformed.loc[bmi_outliers, 'BMI'])

Identified 7 BMI outliers at threshold 3.5:


792     43.0
797     43.7
3878    44.0
4444     9.3
4771    43.5
4820    43.8
6398     8.3
Name: BMI, dtype: float64

In [13]:
df_transformed = df_transformed.copy().drop(index=bmi_outliers).reset_index(drop=True)

In [14]:
df_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 7951 entries, 0 to 7950
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   PatientID                   7951 non-null   int64  
 1   Age                         7951 non-null   int64  
 2   Gender                      7951 non-null   str    
 3   Ethnicity                   7951 non-null   str    
 4   Hospital ID                 7951 non-null   str    
 5   Height (m)                  7951 non-null   float64
 6   Smoker                      7951 non-null   bool   
 7   BMI                         7951 non-null   float64
 8   Weight (kg)                 7951 non-null   float64
 9   Adjusted Weight (kg)        7951 non-null   float64
 10  Has Diabetes                7951 non-null   int64  
 11  Has Hypertension            7951 non-null   int64  
 12  Exercise Frequency          7951 non-null   str    
 13  Diet Type                   7951 non-null   

In [15]:
df_transformed.to_csv("../data/processed/healthcare_readmissions_dataset_train_no_outliers.csv", index=False)

In [16]:

RAW_TRAIN_PATH = Path("../data/processed/healthcare_readmissions_dataset_train_no_outliers.csv")
DATASET_VERSION = "v1.0"
EXPERIMENT_NAME = "readmissions-model-exp"

log_dataset_to_gcs(
    DATASET_LOCAL_PATH=RAW_TRAIN_PATH,
    VERSION_ID=DATASET_VERSION,
    BUCKET_ROOT_URI=BUCKET_ROOT_URI,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    log_experiment=True,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    description="Training data without outliers",
    tags=["no_outliers"],
    resume_run=True,
)


Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v1.0/train.csv
Uploaded: gs://readmissions_bucket_v2/datasets/readmissions/v1.0/manifest.json


Logged dataset version to Vertex Experiments: readmissions-model-exp / readmissions-data-v1-0


In [17]:
DATASET_VERSION = "v1.0"
dataset, DATASET_GCS_URI = register_vertex_dataset(
    dataset_version=DATASET_VERSION,
    bucket_root_uri=BUCKET_ROOT_URI,
    project_id=PROJECT_ID,
    location=LOCATION,
)
DATASET_RESOURCE_NAME = dataset.resource_name


Reusing existing dataset: projects/182027088454/locations/us-central1/datasets/1453729744024502272
Dataset registered : projects/182027088454/locations/us-central1/datasets/1453729744024502272
GCS source         : gs://readmissions_bucket_v2/datasets/readmissions/v1.0/train.csv


## **XGBoost Experimentation**
- **Baseline**: Default hyperparameters, no oversampling.  
- **Experiment 1**: Default hyperparameters, with oversampling.  
- **Experiment 2**: Hyperparameter tuning with oversampling (e.g. `n_estimators=200`, `max_depth=5`).

### Baseline — XGBoost, No Oversampling

Default XGBoost hyperparameters on the imbalanced training set. Establishes the performance floor before any class-balance intervention.


In [18]:
NO_OVERSAMPLE_TRAINING_PIPELINE_JSON = "../vertex/pipelines/readmissions_training_no_oversample_pipeline.json"

compiler.Compiler().compile(
    pipeline_func=training_pipeline_no_oversample,
    package_path=NO_OVERSAMPLE_TRAINING_PIPELINE_JSON,
)

print(f"Pipeline compiled to {NO_OVERSAMPLE_TRAINING_PIPELINE_JSON}")


Pipeline compiled to ../vertex/pipelines/readmissions_training_no_oversample_pipeline.json


In [19]:
# Submit training pipeline, then log the run (with eval metrics) to Vertex Experiments.
MODEL_VERSION = "v0"
EXPERIMENT_NAME = "xgboost-exp"

job = aiplatform.PipelineJob(
    display_name=f"readmissions-training-{DATASET_VERSION}-{MODEL_VERSION}-baseline-run-1",
    template_path=NO_OVERSAMPLE_TRAINING_PIPELINE_JSON,
    pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
    parameter_values={
        "dataset_gcs_uri": DATASET_GCS_URI,
        "dataset_version": DATASET_VERSION,
        "model_type": "xgboost",
        "hyperparams_json": "{}",
    },
)

job.submit()
print(f"Pipeline submitted: {job.display_name}")

log_pipeline_run(
    pipeline_job=job,
    dataset_version=DATASET_VERSION,
    training_dataset_path=DATASET_GCS_URI,
    model_version=MODEL_VERSION,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    wait_for_completion=True,
)


Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-no-oversample-20260524115315
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-no-oversample-20260524115315')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/readmissions-training-pipeline-no-oversample-20260524115315?project=182027088454
Pipeline submitted: readmissions-training-v1.0-v0-baseline-run-1


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/xgboost-exp-pipeline-run-v0-20260524185315 to Experiment: xgboost-exp


Waiting for pipeline to complete...
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-no-oversample-20260524115315
Logged eval metrics: {'val_precision': 0.8, 'val_recall': 0.6521739130434783, 'val_pr_auc': 0.8430751026773832, 'val_roc_auc': 0.9552212486912438, 'val_f1': 0.718562874251497}
Logged pipeline run to Vertex Experiments: xgboost-exp / pipeline-run-v0-20260524185315


### Experiment 1 — XGBoost with SMOTE Oversampling

Same default hyperparameters as baseline, but the training split is oversampled via SMOTE before fitting. Isolates the effect of class balancing on recall.


In [20]:
job = aiplatform.PipelineJob(
    display_name=f"readmissions-training-{DATASET_VERSION}-{MODEL_VERSION}-oversampling-experiment-1-run-1",
    template_path=TRAINING_PIPELINE_JSON,
    pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
    parameter_values={
        "dataset_gcs_uri": DATASET_GCS_URI,
        "dataset_version": DATASET_VERSION,
        "model_type": "xgboost",
        "hyperparams_json": "{}",
    },
)

job.submit()
print(f"Pipeline submitted: {job.display_name}")

log_pipeline_run(
    pipeline_job=job,
    dataset_version=DATASET_VERSION,
    training_dataset_path=DATASET_GCS_URI,
    model_version=MODEL_VERSION,
    PROJECT_ID=PROJECT_ID,
    LOCATION=LOCATION,
    EXPERIMENT_NAME=EXPERIMENT_NAME,
    wait_for_completion=True,
)

Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524115324
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524115324')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/readmissions-training-pipeline-20260524115324?project=182027088454
Pipeline submitted: readmissions-training-v1.0-v0-oversampling-experiment-1-run-1


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/xgboost-exp-pipeline-run-v0-20260524185325 to Experiment: xgboost-exp


Waiting for pipeline to complete...
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/readmissions-training-pipeline-20260524115324
Logged eval metrics: {'val_precision': 0.7473309608540926, 'val_recall': 0.7608695652173914, 'val_pr_auc': 0.8439029493008561, 'val_roc_auc': 0.9575356808287872, 'val_f1': 0.7540394973070018}
Logged pipeline run to Vertex Experiments: xgboost-exp / pipeline-run-v0-20260524185325


### Experiment 2 — XGBoost + SMOTE, Hyperparameter Grid Search

Grid search over `n_estimators`, `max_depth`, and `learning_rate` with SMOTE oversampling. All combinations are submitted in parallel to Vertex AI and results are compared in the table below.


In [21]:
import json
import itertools
from datetime import datetime, timezone

HPT_EXPERIMENT_NAME = "xgboost-hpt-exp"
DATASET_VERSION = "v1.0"
DATASET_GCS_URI = f"{BUCKET_ROOT_URI}/datasets/readmissions/{DATASET_VERSION}/train.csv"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

# Define the hyperparameter search grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
}

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Submitting {len(combinations)} pipeline runs...")

# Submit all jobs up front — they run in parallel on Vertex AI infrastructure
hpt_jobs = []
for i, hyperparams in enumerate(combinations):
    model_version = f"hpt-v{i}"
    job = aiplatform.PipelineJob(
        display_name=f"readmissions-hpt-{DATASET_VERSION}-{model_version}-{RUN_TIMESTAMP}",
        job_id=f"hpt-{i}-{RUN_TIMESTAMP}",
        template_path=TRAINING_PIPELINE_JSON,
        pipeline_root=f"{BUCKET_ROOT_URI}/pipelines",
        parameter_values={
            "dataset_gcs_uri": DATASET_GCS_URI,
            "dataset_version": DATASET_VERSION,
            "model_type": "xgboost",
            "hyperparams_json": json.dumps(hyperparams),
        },
    )
    job.submit()
    hpt_jobs.append((job, hyperparams, model_version))
    print(f"  [{i+1}/{len(combinations)}] Submitted: {model_version} | {hyperparams}")

print(f"\nAll {len(hpt_jobs)} jobs submitted. Run the next cell to wait and log results.")

Submitting 8 pipeline runs...
Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/hpt-0-20260524185334
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/hpt-0-20260524185334')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/hpt-0-20260524185334?project=182027088454
  [1/8] Submitted: hpt-v0 | {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05}
Creating PipelineJob
PipelineJob created. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/hpt-1-20260524185334
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/182027088454/locations/us-central1/pipelineJobs/hpt-1-20260524185334')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/hpt-1-20260524185334?project=18202708

In [22]:
# Wait for each job to finish and log params + eval metrics to the experiment.
# Jobs already run in parallel on Vertex AI; this cell just blocks until each completes.
for job, hyperparams, model_version in hpt_jobs:
    print(f"Waiting for {model_version}...")
    log_pipeline_run(
        pipeline_job=job,
        dataset_version=DATASET_VERSION,
        training_dataset_path=DATASET_GCS_URI,
        model_version=model_version,
        PROJECT_ID=PROJECT_ID,
        LOCATION=LOCATION,
        EXPERIMENT_NAME=HPT_EXPERIMENT_NAME,
        wait_for_completion=True,
        eval_task_name="evaluate-model",
        hyperparams=hyperparams,
    )

print("All HPT runs logged.")

Waiting for hpt-v0...


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/xgboost-hpt-exp-pipeline-run-hpt-v0-20260524185341 to Experiment: xgboost-hpt-exp


Waiting for pipeline to complete...
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/hpt-0-20260524185334
Logged eval metrics: {'val_precision': 0.5966183574879227, 'val_recall': 0.894927536231884, 'val_pr_auc': 0.8464976021965004, 'val_roc_auc': 0.9549250564831652, 'val_f1': 0.7159420289855073}
Logged pipeline run to Vertex Experiments: xgboost-hpt-exp / pipeline-run-hpt-v0-20260524185341
Waiting for hpt-v1...


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/xgboost-hpt-exp-pipeline-run-hpt-v1-20260524185350 to Experiment: xgboost-hpt-exp


Waiting for pipeline to complete...
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/hpt-1-20260524185334
Logged eval metrics: {'val_precision': 0.631979695431472, 'val_pr_auc': 0.8649755455981022, 'val_recall': 0.9021739130434783, 'val_roc_auc': 0.961459194357194, 'val_f1': 0.7432835820895523}
Logged pipeline run to Vertex Experiments: xgboost-hpt-exp / pipeline-run-hpt-v1-20260524185350
Waiting for hpt-v2...


...

Waiting for pipeline to complete...
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/hpt-5-20260524185334
Logged eval metrics: {'val_precision': 0.6630434782608695, 'val_recall': 0.8840579710144928, 'val_pr_auc': 0.8803185449348767, 'val_roc_auc': 0.9662850608916074, 'val_f1': 0.7577639751552795}
Logged pipeline run to Vertex Experiments: xgboost-hpt-exp / pipeline-run-hpt-v5-20260524185423
Waiting for hpt-v6...


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/xgboost-hpt-exp-pipeline-run-hpt-v6-20260524185431 to Experiment: xgboost-hpt-exp


Waiting for pipeline to complete...
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/hpt-6-20260524185334
Logged eval metrics: {'val_precision': 0.6781609195402298, 'val_recall': 0.855072463768116, 'val_pr_auc': 0.8720409251423978, 'val_roc_auc': 0.9641538546316194, 'val_f1': 0.7564102564102564}
Logged pipeline run to Vertex Experiments: xgboost-hpt-exp / pipeline-run-hpt-v6-20260524185431
Waiting for hpt-v7...


Associating projects/182027088454/locations/us-central1/metadataStores/default/contexts/xgboost-hpt-exp-pipeline-run-hpt-v7-20260524185439 to Experiment: xgboost-hpt-exp


Waiting for pipeline to complete...
PipelineJob run completed. Resource name: projects/182027088454/locations/us-central1/pipelineJobs/hpt-7-20260524185334
Logged eval metrics: {'val_precision': 0.7133757961783439, 'val_recall': 0.8115942028985508, 'val_pr_auc': 0.8637606768410283, 'val_roc_auc': 0.9636468837824433, 'val_f1': 0.7593220338983051}
Logged pipeline run to Vertex Experiments: xgboost-hpt-exp / pipeline-run-hpt-v7-20260524185439
All HPT runs logged.


### HPT Results — Ranked Comparison

Fetch all runs from the `xgboost-hpt-exp` experiment and display them ranked by `val_roc_auc`. Use this table to select the best hyperparameter combination for a final retrain.


In [23]:
# Fetch all runs from the experiment and display a ranked comparison table.
aiplatform.init(project=PROJECT_ID, location=LOCATION, experiment=HPT_EXPERIMENT_NAME)
runs_df = aiplatform.get_experiment_df(experiment=HPT_EXPERIMENT_NAME)

metric_cols = sorted(c for c in runs_df.columns if c.startswith("metric."))
param_cols = sorted(c for c in runs_df.columns if c.startswith("param."))

sort_col = "metric.val_roc_auc" if "metric.val_roc_auc" in runs_df.columns else None
result = runs_df[["run_name"] + param_cols + metric_cols].reset_index(drop=True)
if sort_col:
    result = result.sort_values(sort_col, ascending=False).reset_index(drop=True)
else:
    print("Warning: 'metric.val_roc_auc' not found — pipeline runs may have failed or metrics were not logged.")
    print(f"Available columns: {list(runs_df.columns)}")

display(result)


,run_name,param.dataset_version,param.hp_learning_rate,param.hp_max_depth,param.hp_n_estimators,param.model_version,param.pipeline_job_display_name,param.pipeline_resource_name,param.training_dataset_path,metric.val_f1,metric.val_pr_auc,metric.val_precision,metric.val_recall,metric.val_roc_auc
0,pipeline-run-hpt-v5-20260524185423,v1.0,0.10,3.0,200.0,hpt-v5,readmissions-hpt-v1.0-hpt-v5-20260524185334,projects/182027088454/locations/us-central1/pi...,gs://readmissions_bucket_v2/datasets/readmissi...,0.757764,0.880319,0.663043,0.884058,0.966285
1,pipeline-run-hpt-v5-20260524172704,v1.0,0.10,3.0,200.0,hpt-v5,readmissions-hpt-v1.0-hpt-v5-20260524172612,projects/182027088454/locations/us-central1/pi...,gs://readmissions_bucket_v2/datasets/readmissi...,0.757764,0.880319,0.663043,0.884058,0.966285
2,pipeline-run-hpt-v3-20260524185406,v1.0,0.10,5.0,100.0,hpt-v3,readmissions-hpt-v1.0-hpt-v3-20260524185334,projects/182027088454/locations/us-central1/pi...,gs://readmissions_bucket_v2/datasets/readmissi...,0.755267,0.873972,0.683284,0.844203,0.965300
3,pipeline-run-hpt-v3-20260524172646,v1.0,0.10,5.0,100.0,hpt-v3,readmissions-hpt-v1.0-hpt-v3-20260524172612,projects/182027088454/locations/us-central1/pi...,gs://readmissions_bucket_v2/datasets/readmissi...,0.755267,0.873972,0.683284,0.844203,0.965300
4,pipeline-run-hpt-v6-20260524185431,v1.0,0.05,5.0,200.0,hpt-v6,readmissions-hpt-v1.0-hpt-v6-20260524185334,projects/182027088454/locations/us-central1/pi...,gs://readmissions_bucket_v2/datasets/readmissi...,0.756410,0.872041,0.678161,0.855072,0.964154
5,pipeline-run-hpt-v6-20260524172713,v1.0,0.05,5.0,200.0,hpt-v6,readmissions-hpt-v1.0-hpt-v6-20260524172612,projects/182027088454/locations/us-central1/pi...,gs://readmissions_bucket_v2/datasets/readmissi...,0.756410,0.872041,0.678161,0.855072,0.964154
6,pipeline-run-hpt-v7-20260524185439,v1.0,0.10,5.0,200.0,hpt-v7,readmissions-hpt-v1.0-hpt-v7-20260524185334,projects/182027088454/locations/us-central1/pi...,gs://readmissions_bucket_v2/datasets/readmissi...,0.759322,0.863761,0.713376,0.811594,0.963647
7,pipeline-run-hpt-v7-20260524172722,v1.0,0.10,5.0,200.0,hpt-v7,readmissions-hpt-v1.0-hpt-v7-20260524172612,projects/182027088454/locations/us-central1/pi...,gs://readmissions_bucket_v2/datasets/readmissi...,0.759322,0.863761,0.713376,0.811594,0.963647
8,pipeline-run-hpt-v4-20260524185415,v1.0,0.05,3.0,200.0,hpt-v4,readmissions-hpt-v1.0-hpt-v4-20260524185334,projects/182027088454/locations/us-central1/pi...,gs://readmissions_bucket_v2/datasets/readmissi...,0.745509,0.870071,0.635204,0.902174,0.962869
9,pipeline-run-hpt-v4-20260524172655,v1.0,0.05,3.0,200.0,hpt-v4,readmissions-hpt-v1.0-hpt-v4-20260524172612,projects/182027088454/locations/us-central1/pi...,gs://readmissions_bucket_v2/datasets/readmissi...,0.745509,0.870071,0.635204,0.902174,0.962869
